# Projekt: Der Helpdesk der Datenwald-Akademie
### Unterrichtsmaterial | Julia Brutskaya

---
**Kontext:** Abschlussprojekt der Woche Wahrscheinlichkeitsrechnung (Notebooks 8 bis 12)<br>
**Bibliotheken:** `numpy`, `pandas`, `matplotlib`<br>
**Daten:** `csv & excel Dateien/helpdesk_tickets.csv`

<img src="png%20Dateien/mats_maulwurf_huegel.png" alt="Mats" width="120"> <img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="120"> <img src="png%20Dateien/etta_elster.png" alt="Etta" width="120"> <img src="png%20Dateien/fiete_fuchs.png" alt="Fiete" width="120">

Die Datenwald-Akademie hat einen eigenen IT-Helpdesk. Er ist montags bis freitags von 8 bis 18 Uhr besetzt. 
Jeden Montagvormittag klagt das Team: „Wir ersaufen in Tickets!"

Die Leitung fragt: **Wie viele Personen brauchen wir am Montagvormittag?** 
Und nebenbei: Welcher Kanal funktioniert am besten, welche Tickets dauern lange, und welche sind auffällig?

Du hast die Tickets von acht Wochen. Am Ende schreibst du einen **One-Pager** für die Leitung.

**Die Daten:**

| Spalte | Bedeutung |
|---|---|
| `ticket_id` | Nummer des Tickets |
| `zeitpunkt` | Eingang des Tickets (Datum und Uhrzeit, als Text) |
| `kanal` | Telefon, Mail oder Portal |
| `kategorie` | Passwort, Software, Hardware oder Netzwerk |
| `bearbeitung_min` | Bearbeitungszeit in Minuten |
| `erstloesung` | `True`, wenn das Ticket beim ersten Kontakt gelöst wurde |
| `eskaliert` | `True`, wenn das Ticket an die Fachabteilung ging |

**Ablauf und Werkzeuge:**

| Teil | Frage | Werkzeuge aus |
|---|---|---|
| 1 | Daten laden und prüfen | Statistik 7, Notebook 11 |
| 2 | Wahrscheinlichkeiten und Bedingen | Notebook 8 und 9 |
| 3 | Tickets über die Zeit | Notebook 11 |
| 4 | Die Bearbeitungszeit beschreiben | Notebook 12 |
| 5 | Auffällige Tickets finden | Notebook 11 und 12 |
| 6 | Wie sicher sind die Zahlen? | Notebook 12 |
| 7 | Monte-Carlo: der Montagvormittag | Notebook 10 und 12 |
| 8 | One-Pager für die Leitung | Statistik 7 |

Alle Werkzeuge kennst du schon. Neu ist nur, dass du sie selbst kombinieren musst. Lies jede Aufgabe genau: Oft steht im Hinweis, welches Werkzeug passt.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ordner = "csv & excel Dateien/"
GRUEN, ROT, GELB, GRAU = "#2e9e44", "#d62828", "#e0a800", "#c8c8c8"

---
## Teil 1 – Daten laden und prüfen

- **a)** Lies `helpdesk_tickets.csv` als `tickets` ein. Welche Datentypen haben die Spalten? Welche Spalte hat den falschen Typ?
- **b)** Wandle `zeitpunkt` mit `pd.to_datetime` um. Von wann bis wann reichen die Daten, und wie viele Tickets sind es?
- **c)** Lege die Spalten `wochentag` (Name des Tages), `stunde` und `mo_vormittag` an. `mo_vormittag` ist `True` für Montag zwischen 8 und 12 Uhr.
    *Hinweis: `.dt.dayofweek == 0`.*
- **d)** Prüfe kurz die Datenqualität: fehlende Werte, doppelte `ticket_id`, unmögliche Bearbeitungszeiten (≤ 0).

In [2]:
# a)


In [3]:
# b)


In [4]:
# c)


In [5]:
# d)


---
## Teil 2 – Wahrscheinlichkeiten und Bedingen

<img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="110">

> **Herr Bauchgefühl:** „Telefon ist einfach der beste Kanal. Da wird fast alles sofort gelöst. Wir sollten alle Leute ans Telefon setzen!"

- **a)** Wie verteilen sich die Tickets auf Kanäle und Kategorien? Gib die Anteile an.
- **b)** Berechne P(Erstlösung | Kanal) für alle drei Kanäle mit `pd.crosstab(..., normalize="index")`. Stimmt Herr Bauchgefühl zunächst?
- **c)** Sind die Ereignisse „Telefon" und „Passwort" unabhängig? Vergleiche P(Telefon ∩ Passwort) mit P(Telefon) · P(Passwort). Zeige dazu die Kategorien pro Kanal.
- **d)** Berechne die Erstlösungsquote pro Kategorie **und** Kanal (`pd.crosstab` mit `values=` und `aggfunc="mean"`). Ist Telefon in jeder Kategorie besser? Was bedeutet das für die Aussage von Herrn Bauchgefühl?
- **e)** Herr Bauchgefühl: „39 % der eskalierten Tickets sind Hardware-Tickets. Also eskalieren 39 % aller Hardware-Tickets." Prüfe beide Wahrscheinlichkeiten.

In [6]:
# a)


In [7]:
# b)


In [8]:
# c)


In [9]:
# d)


In [10]:
# e)


---
## Teil 3 – Tickets über die Zeit

<img src="png%20Dateien/mats_maulwurf_huegel.png" alt="Mats" width="110">

**Mats:** „Poisson braucht eine konstante Rate. Am Montagmorgen kommen aber viel mehr Tickets als am Freitagnachmittag."

- **a)** Zähle die Tickets pro Stunde mit `resample("h").size()`. Warum ist der Mittelwert über alle Stunden irreführend? Filtere auf Montag bis Freitag, 8 bis 18 Uhr.
- **b)** Erstelle eine Kreuztabelle Wochentag × Stunde und zeichne sie als Heatmap mit `imshow` (wie in Notebook 6). Wo ist es am vollsten, wo am ruhigsten?
- **c)** Schätze λ̂ (Tickets pro Stunde) für drei Zeitfenster: Montag 8 bis 12 Uhr, nachmittags ab 16 Uhr und die restlichen Öffnungsstunden. Lege dafür in den Stunden-Daten eine Spalte `fenster` an. *Hinweis: `np.select` oder zwei Masken.*
- **d)** Prüfe für die „restlichen Öffnungsstunden", ob Poisson passt: Liegen Mittelwert und Varianz nah beieinander? Vergleiche den Anteil der Stunden ohne Ticket mit e^(−λ̂).

In [11]:
# a)


In [12]:
# b)


In [13]:
# c)


In [14]:
# d)


---
## Teil 4 – Die Bearbeitungszeit beschreiben

> **Herr Bauchgefühl:** „Ein Ticket dauert im Schnitt knapp 18 Minuten. Das reicht für die Planung."

- **a)** Berechne Mittelwert, Median, Schiefe und Wölbung von `bearbeitung_min`. Welche Form hat die Verteilung?
- **b)** Zeichne das Histogramm mit Median (grün) und Mittelwert (rot) als `axvline`. Zeige daneben das Histogramm von `np.log10(bearbeitung_min)`. Wie groß ist die Schiefe danach?
- **c)** Schreibe **eine Kette**, die pro Kategorie `anzahl`, `median`, `p90` und `anteil_ueber_30` (Anteil der Tickets über 30 Minuten) berechnet und nach `median` sortiert. *Hinweis: `assign`, dann `groupby` und `agg`.*
- **d)** Was antwortest du Herrn Bauchgefühl?

In [15]:
# a)


In [16]:
# b)


In [17]:
# c)


In [18]:
# d)


---
## Teil 5 – Auffällige Tickets finden

<img src="png%20Dateien/etta_elster.png" alt="Etta" width="110">

**Etta:** „Ein Passwort-Ticket mit 75 Minuten ist viel seltsamer als ein Hardware-Ticket mit 75 Minuten. Ich vergleiche jedes Ticket nur mit seiner eigenen Kategorie."

- **a)** Berechne zuerst den z-Wert über **alle** Tickets ohne Gruppen: `z_roh = (x − Mittelwert) / Standardabweichung`. Welche Tickets liegen über 3? Aus welchen Kategorien kommen sie?
- **b)** Die Daten sind rechtsschief, z-Werte setzen aber eine Glocke voraus. Berechne deshalb `z_log`: den z-Wert von `np.log10(bearbeitung_min)`, **pro Kategorie** mit `groupby().transform`.
- **c)** Welche Tickets haben |z_log| > 3? Sortiere nach z_log. Welche fehlten in a)?
- **d)** Sind die gefundenen Tickets Fehler? Was würdest du mit ihnen tun?

In [19]:
# a)


In [20]:
# b)


In [21]:
# c)


In [22]:
# d)


---
## Teil 6 – Wie sicher sind die Zahlen?

<img src="png%20Dateien/mats_maulwurf_huegel.png" alt="Mats" width="110">

**Mats:** „Über das Portal kamen nur gut 300 Tickets. Da glaube ich der Quote weniger als beim Mail-Kanal."

- **a)** Schätze die Erstlösungsquote p̂ für jeden Kanal.
- **b)** Bootstrap: Ziehe für jeden Kanal 1000-mal eine Stichprobe mit Zurücklegen in gleicher Größe und berechne jeweils die Quote. Gib den Bereich an, in dem die mittleren 95 % liegen. *Hinweis: `sample(frac=1, replace=True, random_state=i)`, dann `quantile([0.025, 0.975])`.*
- **c)** Bei welchem Kanal ist der Bereich am breitesten? Passt das zu Mats?
- **d)** Überschneiden sich die Bereiche von Mail und Portal? Was sagt das?

In [23]:
# a)


In [24]:
# b)


In [25]:
# c)


In [26]:
# d)


---
## Teil 7 – Monte-Carlo: der Montagvormittag

<img src="png%20Dateien/fiete_fuchs.png" alt="Fiete Fuchs" width="110">

**Fiete Fuchs:** „Ich wette, zwei Leute schaffen den Montagvormittag locker. Im Schnitt sind das doch nur 480 Minuten Arbeit, und zwei Leute haben 2 · 240 Minuten."

Wir simulieren 10 000 Montagvormittage (8 bis 12 Uhr = 4 Stunden). Jede Person schafft 240 Minuten Arbeit.

1. **Anzahl der Tickets:** Poisson mit λ = λ̂(Montag 8–12) · 4, aus Teil 3c.
2. **Bearbeitungszeit jedes Tickets:** zufällig aus den **echten** Bearbeitungszeiten gezogen (mit Zurücklegen). So brauchen wir keine Annahme über die Verteilung.
3. **Arbeit** = Summe der Bearbeitungszeiten.

- **a)** Simuliere mit `rng = np.random.default_rng(13)`: die Anzahl mit `rng.poisson(..., size=10_000)` und für jeden Vormittag die Arbeit mit `rng.choice(zeiten, size=anzahl).sum()` (List Comprehension). Speichere alles in einem DataFrame `sim` mit den Spalten `anzahl` und `arbeit_min`.
- **b)** Wie viel Arbeit fällt im Mittel an? Wie groß ist das P90?
- **c)** Berechne für 1 bis 4 Personen die Wahrscheinlichkeit, dass die Arbeit nicht reicht (Arbeit > Personen · 240). Hat Fiete recht?
- **d)** Zeichne die ECDF der Arbeit mit senkrechten Linien bei 480 und 720 Minuten. Speichere die Grafik mit `fig.savefig("png Dateien/helpdesk_montag.png", dpi=120, bbox_inches="tight")` für den One-Pager.

In [27]:
# a)


In [28]:
# b)


In [29]:
# c)


In [30]:
# d)


---
## Teil 8 – One-Pager für die Leitung

Schreibe in die Markdown-Zelle darunter einen One-Pager wie in Statistik 7. Höchstens eine Bildschirmseite.

- **Kernaussage:** ein Satz, den die Leitung sich merkt
- **Diagramm:** deine Grafik aus Teil 7 (`<img src="png%20Dateien/helpdesk_montag.png" width="500">`)
- **Drei Belege:** die wichtigsten Zahlen aus Teil 2 bis 7
- **Grenzen der Aussage:** Welche Annahmen stecken in der Simulation? Was wissen wir nicht?
- **Empfehlung:** eine konkrete nächste Maßnahme

### One-Pager: Personalbedarf am Montagvormittag

**Kernaussage:** …

**Diagramm:** …

**Belege:**
1. …
2. …
3. …

**Grenzen:** …

**Empfehlung:** …

---
## Checkliste zum Abschluss

| Werkzeug | benutzt in Teil |
|---|---|
| `to_datetime`, `.dt.dayofweek`, `.dt.hour`, `.dt.day_name()` | 1, 3 |
| `value_counts(normalize=True)`, `crosstab(normalize="index")` | 2 |
| Unabhängigkeit P(A ∩ B) = P(A) · P(B), Bedingung vertauscht | 2 |
| `resample("h").size()`, Poisson-Check | 3 |
| `skew`, `kurt`, `quantile`, `np.log10` | 4, 5 |
| `assign` und Ketten | 4 |
| `groupby().transform` für z-Werte | 5 |
| Bootstrap mit `sample(frac=1, replace=True)` | 6 |
| `rng.poisson`, `rng.choice`, Monte-Carlo, ECDF | 7 |
| One-Pager mit Grenzen der Aussage | 8 |

---
*Material erstellt von Julia Brutskaya*